In [3]:
import json
import pandas as pd
from pathlib import Path

LOG_PATH = Path("E2.log")
OUTPUT_PATH = Path("scene_evaluation_responses_E2.csv")

rows = []
with open(LOG_PATH, "r") as f:
    for line in f:
        if "scene_evaluation_completed" in line:
            try:
                # 로그 한 줄에서 JSON 파트 추출
                log_data = json.loads(line.split(" - INFO - ")[1])
                details = log_data["details"]

                scene_id = details["sceneId"]
                system_type = details["systemType"]
                system_name = details["systemName"]
                responses = details["responses"]

                # responses = { "1": "yes", "2": "no", ... }
                for q_id, ans in responses.items():
                    rows.append(
                        {
                            "sceneId": scene_id,
                            "systemType": system_type,
                            "systemName": system_name,
                            "questionId": int(q_id),
                            "answer": ans,
                        }
                    )
            except Exception as e:
                print("Parse error:", e)

# DataFrame으로 변환
df = pd.DataFrame(rows)

# CSV 저장
df.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Saved {len(df)} responses to {OUTPUT_PATH}")
print(df.head(20))

✅ Saved 456 responses to scene_evaluation_responses_E2.csv
    sceneId systemType systemName  questionId   answer
0         1    system1   Baseline           1  unknown
1         1    system1   Baseline           2      yes
2         1    system1   Baseline           3       no
3         1    system1   Baseline           4      yes
4         1    system1   Baseline           5      yes
5         1    system1   Baseline           6  unknown
6         1    system1   Baseline           7      yes
7         1    system1   Baseline           8  unknown
8         1    system1   Baseline           9       no
9         1    system1   Baseline          10      yes
10        1    system1   Baseline          11      yes
11        1    system1   Baseline          12       no
12        1    system1   Baseline          13       no
13        1    system1   Baseline          14      yes
14        1    system1   Baseline          15  unknown
15        1    system1   Baseline          16      yes
16    

In [50]:
import pandas as pd
from scipy import stats

# ====== 1) Load + rater 구분 ======
files = {
    # "E1": "scene_evaluation_responses_E1.csv",
    # "E2": "scene_evaluation_responses_E2.csv",
    "E3": "scene_evaluation_responses_E3.csv",
    "E4": "scene_evaluation_responses_E4.csv",
}

dfs = []
for rater, path in files.items():
    df_tmp = pd.read_csv(path)
    df_tmp["rater_id"] = rater
    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)

df_filtered = df.copy()

# ====== 3) sceneId × systemType × rater별 yes 비율 ======
yes_ratio_df = (
    df_filtered.groupby(["sceneId", "systemType", "rater_id"])["answer"]
    .apply(lambda x: (x == "yes").mean())
    .reset_index()
    .rename(columns={"answer": "yes_ratio"})
)

# ====== 4) participant_id 매핑 ======
def assign_participant_id(scene_id, system):
    if system == "system1":
        return scene_id
    else:  # system2
        if scene_id <= 6:
            return scene_id + 6
        else:
            return scene_id - 6

yes_ratio_df["participant_id"] = yes_ratio_df.apply(
    lambda r: assign_participant_id(r["sceneId"], r["systemType"]), axis=1
)

# ====== 5) rater별 system1 vs system2 비교 ======
results = {}
for rater, group in yes_ratio_df.groupby("rater_id"):
    pivot_df = group.pivot_table(
        index="participant_id", columns="systemType", values="yes_ratio"
    ).dropna()

    t_stat, p_val = stats.ttest_rel(pivot_df["system1"], pivot_df["system2"])
    results[rater] = {
        "n_pairs": len(pivot_df),
        "t": t_stat,
        "p": p_val,
    }

# ====== 6) 출력 ======
# ====== 6) 출력 ======
for rater, group in yes_ratio_df.groupby("rater_id"):
    pivot_df = group.pivot_table(
        index="participant_id", columns="systemType", values="yes_ratio"
    ).dropna()

    t_stat, p_val = stats.ttest_rel(pivot_df["system1"], pivot_df["system2"])

    mean_std = group.groupby("systemType")["yes_ratio"].agg(["mean", "std"])

    print(f"=== {rater} ===")
    print(f"n={len(pivot_df)}, t={t_stat:.3f}, p={p_val:.4f}")
    print("\n-- Mean / Std by system --")
    print(mean_std)
    print()


=== E3 ===
n=12, t=-0.625, p=0.5446

-- Mean / Std by system --
                mean       std
systemType                    
system1     0.692982  0.132453
system2     0.723684  0.119018

=== E4 ===
n=12, t=-0.072, p=0.9439

-- Mean / Std by system --
                mean       std
systemType                    
system1     0.732456  0.103959
system2     0.736842  0.166436



In [44]:
# ====== 평가자별 평균 후 참가자 단위 비교 ======
acc_avg = (
    yes_ratio_df
    .groupby(["participant_id", "systemType"])["yes_ratio"]
    .mean()
    .reset_index()
)

pivot_avg = acc_avg.pivot(
    index="participant_id", columns="systemType", values="yes_ratio"
).dropna()

t_stat, p_val = stats.ttest_rel(pivot_avg["system1"], pivot_avg["system2"])

print("=== 평균된 Accuracy per participant ===")
print(pivot_avg)
print(f"\nPaired t-test: t = {t_stat:.3f}, p = {p_val:.4f}")


=== 평균된 Accuracy per participant ===
systemType       system1   system2
participant_id                    
1               0.500000  0.815789
2               0.578947  0.842105
3               0.710526  0.657895
4               0.868421  0.815789
5               0.657895  0.710526
6               0.789474  0.842105
7               0.815789  0.868421
8               0.789474  0.552632
9               0.710526  0.447368
10              0.736842  0.842105
11              0.657895  0.631579
12              0.736842  0.736842

Paired t-test: t = -0.358, p = 0.7268


In [45]:
import pandas as pd
from scipy.stats import wilcoxon

# 1) 참가자×시스템별로 두 평가자 점수 평균(또는 .median())
avg_df = (
    yes_ratio_df
    .groupby(["participant_id", "systemType"])["yes_ratio"]
    .mean()                       # 필요시 .median()으로 교체
    .unstack()                    # columns: system1, system2
    .dropna()                     # 두 시스템 모두 존재하는 참가자만
)

# 2) 대응 Wilcoxon (system1 vs system2)
stat, p = wilcoxon(
    avg_df["system1"], avg_df["system2"],
    zero_method="pratt", alternative="less"
)

print(f"n={len(avg_df)}, Wilcoxon W={stat:.3f}, p={p:.6f}")
print(f"mean(system1)={avg_df['system1'].mean():.3f}, mean(system2)={avg_df['system2'].mean():.3f}")


n=12, Wilcoxon W=33.500, p=0.338623
mean(system1)=0.713, mean(system2)=0.730


In [46]:
import pandas as pd
import statsmodels.formula.api as smf

# yes_ratio_df: [participant_id, systemType, rater_id, yes_ratio]

# 혼합효과 모형: systemType 고정효과 + (participant, rater 랜덤효과)
model = smf.mixedlm(
    "yes_ratio ~ systemType", 
    data=yes_ratio_df,
    groups=yes_ratio_df["participant_id"],
    re_formula="~systemType"
).fit(reml=False)

print(model.summary())


                   Mixed Linear Model Regression Results
Model:                   MixedLM        Dependent Variable:        yes_ratio
No. Observations:        48             Method:                    ML       
No. Groups:              12             Scale:                     0.0063   
Min. group size:         4              Log-Likelihood:            36.6918  
Max. group size:         4              Converged:                 Yes      
Mean group size:         4.0                                                
----------------------------------------------------------------------------
                                  Coef.  Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------------------------
Intercept                          0.713    0.029 25.006 0.000  0.657  0.769
systemType[T.system2]              0.018    0.047  0.374 0.708 -0.074  0.109
Group Var                          0.007    0.061                           
Group x systemType[

/Users/jeongin/Library/Python/3.10/lib/python/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [47]:
import pandas as pd
import numpy as np
from scipy import stats

# 참가자별 system1 vs system2 평균 yes_ratio
avg = (
    yes_ratio_df.groupby(["participant_id","systemType"])["yes_ratio"]
                .mean().unstack().dropna()
)

x = avg["system1"].values
y = avg["system2"].values
diff = y - x

# 1) 평균 차이
mean_diff = diff.mean()

# 2) Cohen's d (paired)
cohens_d = mean_diff / diff.std(ddof=1)

# 3) Cliff's delta
n = len(diff)
n_pos = sum(diff > 0)
n_neg = sum(diff < 0)
cliffs_delta = (n_pos - n_neg) / n

print(f"Mean(system2 - system1) = {mean_diff:.3f}")
print(f"Cohen's d (paired)      = {cohens_d:.3f}")
print(f"Cliff's delta           = {cliffs_delta:.3f}")


Mean(system2 - system1) = 0.018
Cohen's d (paired)      = 0.103
Cliff's delta           = 0.000


In [48]:
import pandas as pd
from scipy.stats import wilcoxon

# --- 1) Set A 질문 매핑 ---
q_to_cat_A = {
  1:"attribute", 2:"attribute", 3:"attribute", 4:"attribute",
  5:"attribute", 6:"attribute", 7:"relation",
  8:"attribute", 9:"attribute", 10:"attribute", 11:"attribute", 12:"relation",
  13:"attribute", 14:"attribute", 15:"attribute", 16:"relation", 17:"relation",
  18:"context", 19:"context"
}

# --- 2) Set B 질문 매핑 ---
q_to_cat_B = {
  1:"attribute", 2:"attribute", 3:"attribute", 4:"attribute", 5:"relation",
  6:"attribute", 7:"attribute", 8:"attribute", 9:"relation", 10:"relation",
  11:"attribute", 12:"attribute", 13:"attribute", 14:"relation",
  15:"attribute", 16:"attribute", 17:"relation", 18:"relation",
  19:"context"
}

# --- 3) sceneId에 따라 카테고리 붙이기 ---
def map_category(row):
    sid, qid = row["sceneId"], row["questionId"]
    if sid <= 6:  # Set A
        return q_to_cat_A.get(qid)
    else:        # Set B
        return q_to_cat_B.get(qid)

df_filtered["category"] = df_filtered.apply(map_category, axis=1)

df_filtered["participant_id"] = df_filtered.apply(
    lambda r: assign_participant_id(r["sceneId"], r["systemType"]), axis=1
)

# --- 4) participant × system × category 평균 yes 비율 ---
cat_ratio_df = (
    df_filtered.groupby(["participant_id","systemType","category"])["answer"]
               .apply(lambda x: (x=="yes").mean())
               .reset_index()
)

# --- 5) category별 Wilcoxon (system1 vs system2) ---
results = []
for cat, g in cat_ratio_df.groupby("category"):
    pivot = g.pivot(index="participant_id", columns="systemType", values="answer").dropna()
    if "system1" in pivot and "system2" in pivot and len(pivot) > 0:
        stat, p = wilcoxon(pivot["system1"], pivot["system2"], zero_method="pratt")
        mean_diff = (pivot["system2"] - pivot["system1"]).mean()
        results.append((cat, len(pivot), mean_diff, stat, p))

print("=== Category-wise comparisons ===")
for cat, n, mean_diff, stat, p in results:
    print(f"{cat}: n={n}, Δ={mean_diff:.3f}, W={stat:.3f}, p={p:.4f}")


=== Category-wise comparisons ===
attribute: n=12, Δ=0.019, W=37.000, p=0.9097
context: n=12, Δ=-0.125, W=9.500, p=0.2758
relation: n=12, Δ=0.052, W=28.000, p=0.4087


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:531: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  res = hypotest_fun_out(*samples, **kwds)
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:531: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  res = hypotest_fun_out(*samples, **kwds)
